# Three routes to dBe/dT, dBe/dS, dBe/dCT, dBe/dAT

Analytic, automatic (AD), and finite difference (FD), compared head to head.

The Bolin sensitivity is $B_e = \partial C_T/\partial[\mathrm{CO_2^*}]$ at constant
$A_T, T, S$. We want its four partial derivatives. Three ways to get them:

| route | file | how |
|---|---|---|
| **analytic** | `buffderiv.R` + `dlnK.R` | hand-derived closed form; seacarb supplies K values, we hand-code dK/dT and dK/dS |
| **automatic** | `ad_dual.R` + `ad_be.R` | forward-mode dual numbers + implicit function theorem; nothing hand-differentiated |
| **finite difference** | any | difference `buffsun()` or `buffderiv()$Be` through the full nonlinear `carb()` solve |

The three are genuinely independent: the analytic route hand-codes the *derivatives* and
takes K *values* from seacarb; the AD route hand-codes the K *values* and derives
everything else automatically. If they agree, both are right.

## 1. Why you cannot simply hand `buffsun()` to an AD tool

Two blockers, and understanding them dictates the design.

**(a) `carb()` is a nonlinear solve.** SolveSAPHE iterates on $h$ with `while` loops and
convergence tests. AD *can* differentiate through the iterations, but then you are
differentiating the *algorithm*, not the *function*: the derivative inherits the solver's
convergence error and you pay to tape every iteration.

**(b) seacarb's constants are not differentiable code.** `K1()` and friends end with
`as.numeric()`, `attr()`, `rep()` and subassignment on option strings. Operator-overloading
AD dies at the first `as.numeric()`, which strips the dual class. Tape-based AD (RTMB)
breaks there too.

### The fix: AD + the implicit function theorem

Never differentiate the solver. $h$ is defined implicitly by the alkalinity constraint

$$F(h; T,S,C_T,A_T) = A_c(h,C_T) + A_{nc}(h) - A_T = 0$$

and $B_e$ is an explicit algebraic function $G(h,C_T,T,S)$. So for $Y \in \{T,S,C_T,A_T\}$:

$$\boxed{\;\frac{dB_e}{dY} = G_Y - G_h\,\frac{F_Y}{F_h}\;}$$

Solve $F=0$ once with any (non-differentiable) solver. Then take **one AD pass seeding five
directions** $(T,S,C_T,A_T,h)$ through $F$ and $G$, which hands you $F_Y, F_h, G_Y, G_h$
directly. Combine. Done.

This is the same skeleton as `buffderiv`. The difference is who computes the four partials:
you, or the compiler.

## 2. The AD core: `ad_dual.R` (35 lines)

A **dual number** carries a value $v$ and a vector $d$ of partials with respect to our five
seed directions. Arithmetic is overloaded so the chain rule is applied automatically:

$$(a,\,a')\times(b,\,b') = (ab,\;\; a'b + ab')$$
$$\exp(a,\,a') = (e^a,\;\; e^a a')$$

`seed(v, i)` creates a variable whose derivative with respect to direction $i$ is 1.
Everything downstream then carries its own exact derivative. There is no step size and no
truncation error: this is the chain rule, evaluated in floating point.

The implementation is vectorised: `v` is a numeric vector of length $n$ and `d` is an
$n \times 5$ matrix, so a whole model field is differentiated in one pass.

In [ ]:
source("ad_dual.R")
cat(readLines("ad_dual.R"), sep = "\n")

## 3. The model: `ad_be.R`

Three functions, all written in **plain arithmetic only** (no `as.numeric()`, no `attr()`,
no branching on values) so that duals flow through:

- **`Kall(T, S)`** — an AD-clean re-implementation of the seacarb constant set
  (`k1k2="l"`, `ks="d"`, `kf="dg"`, `b="u74"`, total pH scale). This is the price of AD:
  you must re-implement the K *values*. See the caveat in section 4.
- **`Fres(h, T, S, CT, AT, Pt, Sit)`** — the alkalinity residual, i.e. exactly the equation
  `carb()` inverts, including the `-[HF]` term and the proton/sulfate block.
- **`Bfun(h, T, S, CT, Pt, Sit)`** — $B_e$, the explicit algebraic function `buffsun()`
  evaluates.

and the driver **`be_ad()`**, which is the whole method in three steps:

```r
# 1. refine h so F(h) = 0 to machine precision (plain doubles; no AD needed)
# 2. ONE AD pass, seeding all five directions (T, S, CT, AT, h)
# 3. implicit function theorem:  dBe/dY = G_Y - G_h * F_Y / F_h
```

In [ ]:
source("setup.R")   # library(seacarb) + the buffer functions from ../R
source("ad_be.R")       # AD route (sources ad_dual.R)

cat(paste(grep("^be_ad|^  #|^  for|^  Td|^  Fd|^  Gd|^  F_h|^  dBe|^  list|^\\}",
               readLines("ad_be.R"), value = TRUE), collapse = "\n"))

## 4. Test 1 (load-bearing): does `Kall()` reproduce seacarb's constants?

This is the one test the AD route absolutely must pass. `buffderiv` gets its K values
*from seacarb*, so it cannot drift. The AD route computes them itself, so `Kall()` is a
**second copy** of seacarb's formulas, and second copies rot.

If you adopt AD, this test is not optional.

In [ ]:
scK <- function(S, T) {
  Ks_P0 <- Ks(S = S, T = T, P = 0, ks = "d", warn = "n")
  Kff   <- Kf(S = S, T = T, P = 0, pHscale = "F", kf = "dg", Ks_P0, Ks_P0, warn = "n")
  t2s <- kconv(S = S, T = T, P = 0, kf = "dg", Ks = Ks_P0, Kff = Kff, warn = "n")$ktotal2SWS
  s2c <- kconv(S = S, T = T, P = 0, kf = "dg", Ks = Ks_P0, Kff = Kff, warn = "n")$kSWS2total
  list(K1  = as.numeric(K1 (S=S,T=T,P=0,pHscale="T",k1k2="l",s2c,t2s,warn="n")),
       K2  = as.numeric(K2 (S=S,T=T,P=0,pHscale="T",k1k2="l",s2c,t2s,warn="n")),
       Kb  = as.numeric(Kb (S=S,T=T,P=0,pHscale="T",s2c,t2s,warn="n")),
       Kw  = as.numeric(Kw (S=S,T=T,P=0,pHscale="T",s2c,warn="n")),
       Ksi = as.numeric(Ksi(S=S,T=T,P=0,pHscale="T",s2c,warn="n")),
       K1p = as.numeric(K1p(S=S,T=T,P=0,pHscale="T",s2c,warn="n")),
       K2p = as.numeric(K2p(S=S,T=T,P=0,pHscale="T",s2c,warn="n")),
       K3p = as.numeric(K3p(S=S,T=T,P=0,pHscale="T",s2c,warn="n")),
       Ks  = as.numeric(Ks_P0), Kf = as.numeric(Kff),
       BOR = as.numeric(bor(S = S, b = "u74")))
}

nm <- c("K1","K2","Kb","Kw","Ksi","K1p","K2p","K3p","Ks","Kf","BOR")
worst <- 0
cat(sprintf("%6s%6s", "T", "S")); for (k in nm) cat(sprintf("%9s", k)); cat("\n")
for (ts in list(c(-1,34), c(5,33.5), c(15,30), c(25,35), c(31,37))) {
  a <- Kall(ts[1], ts[2]); b <- scK(ts[2], ts[1])
  cat(sprintf("%6.1f%6.1f", ts[1], ts[2]))
  for (k in nm) {
    rd <- 100*(a[[k]] - b[[k]])/b[[k]]; worst <- max(worst, abs(rd))
    cat(sprintf("%9.1e", rd))
  }
  cat("\n")
}
cat(sprintf("\nWORST Kall() vs seacarb: %.2e %%\n", worst))

The Kb residual (1.1e-11 %) is `S^1.5` roundoff, not a formula difference. Everything else
is bit-exact.

## 5. Test 2: the three routes, head to head

In [ ]:
cases <- data.frame(
  lab = c("polar","temperate","notebook_base","subtropical","warm_salty_gyre","southern_ocean"),
  T   = c(-1, 10, 20, 25, 29, 5),
  S   = c(34, 34.8, 35, 35, 36.5, 33.5),
  CT  = c(2150, 2050, 2000, 2000, 1950, 2200) * 1e-6,
  AT  = c(2300, 2290, 2300, 2300, 2380, 2320) * 1e-6,
  Pt  = c(1.5, 0.5, 0.2, 0.1, 0.05, 1.8) * 1e-6,
  Sit = c(60, 10, 3, 2, 1, 70) * 1e-6, stringsAsFactors = FALSE)

rich <- function(f, x, e) {
  d1 <- (f(x + e)     - f(x - e))     / (2*e)
  d2 <- (f(x + 2*e)   - f(x - 2*e))   / (4*e)
  (4*d1 - d2)/3
}
opt <- list(k1k2 = "l", kf = "dg", ks = "d", pHscale = "T", b = "u74", warn = "n")

wAD <- 0; wFD <- 0
for (i in 1:nrow(cases)) {
  T0<-cases$T[i]; S0<-cases$S[i]; CT0<-cases$CT[i]; AT0<-cases$AT[i]
  P0<-cases$Pt[i]; Si0<-cases$Sit[i]

  ana <- do.call(buffderiv, c(list(15, AT0, CT0, S=S0, T=T0, Pt=P0, Sit=Si0), opt))

  h0 <- 10^(-do.call(carb, c(list(15, AT0, CT0, S=S0, T=T0, P=0, Pt=P0, Sit=Si0), opt))$pH)
  ad <- be_ad(T0, S0, CT0, AT0, P0, Si0, h0)

  Bf <- function(S,T,CT,AT) do.call(buffderiv, c(list(15,AT,CT,S=S,T=T,Pt=P0,Sit=Si0), opt))$Be
  fd <- c(rich(function(x) Bf(S0,x,CT0,AT0), T0,  1e-3),
          rich(function(x) Bf(x,T0,CT0,AT0), S0,  1e-3),
          rich(function(x) Bf(S0,T0,x,AT0),  CT0, CT0*1e-5),
          rich(function(x) Bf(S0,T0,CT0,x),  AT0, AT0*1e-5))

  k <- c("dBe_dT","dBe_dS","dBe_dCT","dBe_dAT")
  A <- unlist(ana[k]); D <- c(ad$dBe_dT, ad$dBe_dS, ad$dBe_dCT, ad$dBe_dAT)
  wAD <- max(wAD, max(abs(100*(D-A)/A))); wFD <- max(wFD, max(abs(100*(fd-A)/A)))

  cat(sprintf("\n--- %s  (Be = %.8f)\n", cases$lab[i], ana$Be))
  cat(sprintf("    %-8s %18s %18s %12s %12s\n","","analytic","AD","AD %diff","FD %diff"))
  for (j in 1:4)
    cat(sprintf("    %-8s %18.10e %18.10e %11.1e%% %11.1e%%\n",
        k[j], A[j], D[j], 100*(D[j]-A[j])/A[j], 100*(fd[j]-A[j])/A[j]))
}
cat(sprintf("\nWORST AD vs analytic : %.2e %%\n", wAD))
cat(sprintf("WORST FD vs analytic : %.2e %%\n", wFD))

**AD reproduces the analytic derivatives to ~1e-11 %, i.e. to roundoff.** These are two
completely independent derivations: one hand-differentiated the constants, the other
hand-coded them and let the chain rule do the rest. Agreement at 1e-11 % means both are
right. FD lands 5 to 6 orders of magnitude worse.

## 6. Test 3: why FD cannot be fixed by choosing a better step

The classic U-curve. Truncation error falls as $\varepsilon^2$; roundoff error grows as
$1/\varepsilon$. The best you can do is the crossover, and it is nowhere near roundoff.

The second column repeats the exercise with $h$ Newton-refined, to separate step-size error
from solver-convergence error.

In [ ]:
S0<-34; T0<--1; CT0<-2150e-6; AT0<-2300e-6; P0<-1.5e-6; Si0<-60e-6
ref <- do.call(buffderiv, c(list(15,AT0,CT0,S=S0,T=T0,Pt=P0,Sit=Si0), opt))$dBe_dT

Be_np <- function(T) do.call(buffderiv,
           c(list(15,AT0,CT0,S=S0,T=T,Pt=P0,Sit=Si0), opt, npolish=0))$Be
Be_p  <- function(T) do.call(buffderiv,
           c(list(15,AT0,CT0,S=S0,T=T,Pt=P0,Sit=Si0), opt, npolish=3))$Be
cd <- function(f, x, e) (f(x+e) - f(x-e)) / (2*e)

cat(sprintf("Polar point. Reference dBe/dT (analytic) = %.10f\n\n", ref))
cat(sprintf("%12s %20s %20s\n", "eps (degC)", "h from carb()", "h Newton-refined"))
for (e in c(1e-1, 1e-2, 1e-3, 1e-4, 1e-5, 1e-6, 1e-7))
  cat(sprintf("%12.0e %19.2e%% %19.2e%%\n", e,
      100*(cd(Be_np,T0,e)-ref)/ref, 100*(cd(Be_p,T0,e)-ref)/ref))

h0 <- 10^(-do.call(carb, c(list(15,AT0,CT0,S=S0,T=T0,P=0,Pt=P0,Sit=Si0), opt))$pH)
g  <- be_ad(T0, S0, CT0, AT0, P0, Si0, h0)
cat(sprintf("\n%12s %19.2e%%   <- no step size exists\n", "AD",
            100*(g$dBe_dT - ref)/ref))

Note the trap this exposes. FD through `carb()` amplifies any error in $h$ by
$B_e / (|dB_e/dT| \cdot 2\varepsilon) \approx 10^5$ at this point. That is how a 6e-7
relative error in $h$ (which is what a *missing fluoride term* in the alkalinity produces)
masquerades as a 1e-3 % error in the derivative, and sends you hunting for a solver
tolerance bug that does not exist. AD and the analytic route are both immune: neither
differences anything.

## 7. Test 4: random sweep over the CMIP6 surface envelope

In [ ]:
set.seed(1); N <- 500
S   <- runif(N, 28, 38)
T   <- runif(N, -1.8, 32)
AT  <- runif(N, 2000, 2450) * 1e-6
CT  <- AT * runif(N, 0.83, 0.99)
Pt  <- runif(N, 0, 2.5) * 1e-6
Sit <- runif(N, 0, 120) * 1e-6

ana <- do.call(buffderiv, c(list(15, AT, CT, S=S, T=T, Pt=Pt, Sit=Sit), opt))
h0  <- 10^(-do.call(carb, c(list(15, AT, CT, S=S, T=T, P=0, Pt=Pt, Sit=Sit), opt))$pH)
ad  <- be_ad(T, S, CT, AT, Pt, Sit, h0)

for (k in c("dBe_dT","dBe_dS","dBe_dCT","dBe_dAT"))
  cat(sprintf("  %-8s AD vs analytic : worst %.2e %%\n", k,
      max(abs(100*(ad[[k]] - ana[[k]]) / ana[[k]]))))
cat(sprintf("\n  Be              : worst %.2e %%\n",
    max(abs(100*(ad$Be - ana$Be)/ana$Be))))

## 8. Test 5: cost

The `carb()` solve is unavoidable in all three routes. What matters is the *marginal* cost
of the derivatives on top of it.

In [ ]:
set.seed(2); N <- 2000
S   <- runif(N, 30, 37); T <- runif(N, -1.8, 30)
AT  <- runif(N, 2100, 2400) * 1e-6; CT <- AT * runif(N, 0.85, 0.97)
Pt  <- runif(N, 0, 2.5) * 1e-6;     Sit <- runif(N, 0, 120) * 1e-6

t_carb <- system.time(cb <- do.call(carb,
            c(list(15, AT, CT, S=S, T=T, P=0, Pt=Pt, Sit=Sit), opt)))[["elapsed"]]
h0 <- 10^(-cb$pH)
t_ana <- system.time(do.call(buffderiv,
            c(list(15, AT, CT, S=S, T=T, Pt=Pt, Sit=Sit), opt)))[["elapsed"]]
t_ad  <- system.time(be_ad(T, S, CT, AT, Pt, Sit, h0))[["elapsed"]]

cat(sprintf("N = %d points\n\n", N))
cat(sprintf("  carb() solve alone (unavoidable)   : %6.3f s\n", t_carb))
cat(sprintf("  buffderiv = carb + analytic derivs : %6.3f s\n", t_ana))
cat(sprintf("  AD derivatives, given h            : %6.3f s\n", t_ad))
cat(sprintf("\n  marginal cost of analytic derivs   : %6.3f s\n", t_ana - t_carb))
cat(sprintf("  marginal cost of AD derivs         : %6.3f s\n", t_ad))
cat(sprintf("\n  FD would need ~8 extra carb() solves per derivative,\n"))
cat(sprintf("  i.e. ~%.0f s for all four. That is the real argument against FD.\n",
            32 * t_carb))

## 9. Summary

| | AD | analytic | finite difference |
|---|---|---|---|
| accuracy vs. exact | ~1e-11 % | ~1e-11 % | ~1e-6 % at best |
| step size to tune | none | none | yes, and no good choice exists |
| effort to derive | none | high | none |
| what you hand-code | the K **values** (`Kall`) | the K **derivatives** (`dlnK`) | nothing |
| stays correct if seacarb changes a coefficient? | **yes, automatically** | **no, silently stale** | yes |
| marginal cost | ~2x analytic | 1x | ~30x |
| failure mode | loud (code will not trace) | **silent (wrong number)** | silent (amplifies solver error) |

**The trade nobody states plainly:** AD does not remove hand-transcription, it *moves* it.
`buffderiv` takes K values from seacarb and hand-codes the derivatives. The AD route
hand-codes the values and derives the rest. That is still a large win, because a wrong K
value is trivially caught (Test 1 above), whereas a wrong dK/dT is not caught by anything
except exactly this kind of cross-check.

**Recommended use:** keep `buffderiv` as the production path (fastest, validated). Keep the
AD version as a **regression test**. Its real value is that it cannot go stale: if a
coefficient in Lueker or Millero is ever updated in seacarb, `dlnK.R` will keep returning
the old derivative and *nothing else in the test suite would notice*. The AD version would
diverge and fail.